In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats

from src.db import read_sql_file
from src.stats import compare_arms

In [2]:
policy = read_sql_file("13_policy_value.sql")
numeric_cols = policy.columns.drop(["segment", "best_email"])
policy[numeric_cols] = policy[numeric_cols].astype(float)
policy

,segment,n_customers,mens_uplift,womens_uplift,best_email,incremental_spend_all_mens,incremental_spend_all_womens,incremental_spend_targeted
0,both,6448.0,1.824986,0.161881,mens,11767.512442,1043.810252,11767.512442
1,mens_only,28818.0,0.682735,0.278354,mens,19675.066230,8021.619454,19675.066230
2,womens_only,28734.0,0.615731,0.631648,womens,17692.408996,18149.772246,18149.772246


In [3]:
totals = pd.Series({
    "(a) no email": 0.0,
    "(b) mens email to all": policy["incremental_spend_all_mens"].sum(),
    "(c) womens email to all": policy["incremental_spend_all_womens"].sum(),
    "(d) targeted by segment": policy["incremental_spend_targeted"].sum(),
})
totals.map("${:,.0f}".format)

(a) no email                    $0
(b) mens email to all      $49,135
(c) womens email to all    $27,215
(d) targeted by segment    $49,592
dtype: str

In [4]:
# (d) differs from (b) only in segments where the womens email has the higher point estimate.
seg = read_sql_file("12_segment_summary_stats.sql")
seg = seg[seg["dimension"] == "past_purchase_category"].astype(
    {"n": int, "spend_mean": float, "spend_var": float}
).set_index(["segment", "arm"])

switched = policy.loc[policy["best_email"] == "womens", ["segment", "n_customers"]]

gain, variance = 0.0, 0.0
for segment, n_customers in switched.itertuples(index=False):
    w, m = seg.loc[(segment, "womens")], seg.loc[(segment, "mens")]
    gain += n_customers * (w["spend_mean"] - m["spend_mean"])
    variance += n_customers**2 * (w["spend_var"] / w["n"] + m["spend_var"] / m["n"])

se = np.sqrt(variance)
z = stats.norm.ppf(0.975)
print("Segments switched to the womens email:", list(switched["segment"]))
print(f"Extra spend of (d) over (b): ${gain:,.0f}  (95% CI ${gain - z*se:,.0f} to ${gain + z*se:,.0f})")

Segments switched to the womens email: ['womens_only']
Extra spend of (d) over (b): $457  (95% CI $-11,283 to $12,198)


In [5]:
summary = read_sql_file("10_arm_summary_stats.sql").set_index("arm").astype(
    {"n": int, "visits": int, "conversions": int, "spend_mean": float, "spend_var": float}
)
spend_effect = compare_arms(summary, "mens", "control").set_index("metric").loc["spend"]

margins = [0.2, 0.3, 0.4, 0.5]
break_even = pd.DataFrame({
    "gross_margin": margins,
    # Maximum cost per email at which sending is still profitable
    "break_even_cost_point_estimate": [m * spend_effect["diff"] for m in margins],
    "break_even_cost_ci_lower_bound": [m * spend_effect["ci_low"] for m in margins],
})
break_even.style.format({
    "gross_margin": "{:.0%}",
    "break_even_cost_point_estimate": "${:.3f}",
    "break_even_cost_ci_lower_bound": "${:.3f}",
})

,gross_margin,break_even_cost_point_estimate,break_even_cost_ci_lower_bound
0,20%,$0.154,$0.097
1,30%,$0.231,$0.146
2,40%,$0.308,$0.194
3,50%,$0.385,$0.243


## Business impact and recommendation

Incremental spend over the 2-week window, relative to sending no email, for 64,000 customers:

| Policy | Incremental spend |
|---|---|
| (a) No email | $0 |
| (b) Mens email to all | $49,135 |
| (c) Womens email to all | $27,215 |
| (d) Targeted by past purchase category | $49,592 |

- Policy (d) differs from (b) only for customers who previously bought only women's merchandise.
  It adds $457 (95% CI −$11,283 to $12,198), which is indistinguishable from zero. The choice to switch is
  based on a small point-estimate difference that is likely noise (winner's curse).
- **Break-even:** at a 30% gross margin, sending the mens email is profitable if each email costs less than
  $0.23 (point estimate), or less than $0.15 at the lower bound of the 95% confidence interval.
- **Recommendation:** send the mens email to all customers. Keep a random holdout group (5–10%) in future
  campaigns to keep measuring incrementality. Test the womens email on new customers as a follow-up hypothesis.
- **Limitations:** spend is revenue, not profit; outcomes cover only 2 weeks; there is no data on email
  costs, unsubscribes or long-term effects; the data is from 2008.